In [0]:
# --------------------------------------------------------------------------------------------------
# SCRIPT: 1.2_diversidade_partidaria_frentes
# LOCAL: 3_gold/src/1_atlas_frentes/
# OBJETIVO: Gerar tabelas de apoio para análise diversidade partidária 
#           (fonte 'gold_atlas_frentes_parlamentares' gerado pelo script 3_gold/1_atlas_frentes/1.1_atlas_frentes_consolidado)
# ENTREGA: Identificação das frentes com maior diversidade partidária (Índice de Herfindahl)
# --------------------------------------------------------------------------------------------------

from pyspark.sql.functions import col, count, pow, sum as _sum

df_atlas = spark.table("gold_atlas_frentes_parlamentares")

df_counts = df_atlas.groupby("id_frente", "nome_frente", "partido").agg(count("*").alias("membros_partido"))

df_total = df_atlas.groupby("id_frente").agg(count("*").alias("total_frente"))


df_indice_herfindahl = df_counts.join(df_total, "id_frente") \
    .withColumn("proporcao", col("membros_partido") / col("total_frente")) \
    .withColumn("sq_proporcao", pow(col("proporcao"),2)) \
    .groupBy("id_frente", "nome_frente") \
    .agg(_sum("sq_proporcao").alias("indice_herfindahl"), count("partido").alias("qtd_partidos")) \
    .orderBy("indice_herfindahl")
    
df_indice_herfindahl.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_atlas_frentes_diversidade")

display(df_indice_herfindahl)